# 02 — EDA: Fall Detection Datasets

**Tarea:** B1 — Exploración de MobiAct v2 + SisFall

Objetivos:
- Visualizar señales de caídas vs ADL (actividades cotidianas)
- Distribución temporal: impacto (pico SVM) → inmovilidad post-caída
- Identificar umbral SVM empírico para la etapa 1 de la cascada
- Tipos de caída (MobiAct: forward, backward, side, syncope)
- Diferencia de señal: sujetos jóvenes (MobiAct) vs ancianos (SisFall)
- Desbalanceo ADL/caídas — motivar SMOTE + class_weight

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.insert(0, str(Path('..').resolve()))
from src.preprocessing.windowing import (
    resample, compute_svm, detect_impact_peaks,
    extract_fall_window, butterworth_lowpass, normalize_subject
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_RAW = Path('../data/raw')
DATA_PROC = Path('../data/processed')
DATA_PROC.mkdir(parents=True, exist_ok=True)

## 1. Dataset Inventory

In [ ]:
# Check dataset availability
for ds in ['MobiAct', 'SisFall']:
    p = DATA_RAW / ds
    if p.exists():
        files = list(p.rglob('*.csv')) + list(p.rglob('*.txt'))
        print(f"{ds}: {len(files)} files found")
    else:
        print(f"{ds}: NOT FOUND — download to data/raw/{ds}/")

## 2. MobiAct v2 — Loading and Exploration

MobiAct v2 — 66 subjects, ~87 Hz, smartphone in pocket.
Fall types: FOL (forward), BSC (backward), SDL (side), SIS (syncope/faint).
ADL types: STD (standing), WAL (walking), JOG (jogging), JUM (jumping), STU (stairs up), STN (stairs down), SCH (squat), SBW (sitting), CHU (chair up), CSI (car step in), CSO (car step out), CUP (cup), LYI (lying).

In [ ]:
MOBIACT_DIR = DATA_RAW / 'MobiAct'
FALL_LABELS_MOBIACT = ['FOL', 'BSC', 'SDL', 'SIS']
ADL_LABELS_MOBIACT = ['STD', 'WAL', 'JOG', 'JUM', 'STU', 'STN', 'SCH', 'SBW', 'CHU', 'CSI', 'CSO', 'CUP', 'LYI']


def load_mobiact_file(filepath: Path) -> pd.DataFrame:
    """Load a single MobiAct CSV file. Columns: timestamp, accel_x, accel_y, accel_z, gyro_x, gyro_y, gyro_z, label."""
    # MobiAct format: header line + data
    df = pd.read_csv(filepath, comment='#')
    return df


def load_mobiact_dataset(base_dir: Path, max_files_per_label: int = None) -> pd.DataFrame:
    """Load all MobiAct files. Returns DataFrame with columns [ax, ay, az, gx, gy, gz, label, subject_id]."""
    all_dfs = []
    for label_dir in sorted(base_dir.iterdir()):
        if not label_dir.is_dir():
            continue
        label = label_dir.name
        csv_files = sorted(label_dir.glob('*.csv'))
        if max_files_per_label:
            csv_files = csv_files[:max_files_per_label]
        for f in csv_files:
            try:
                df = load_mobiact_file(f)
                # Standardize column names
                df.columns = [c.strip().lower() for c in df.columns]
                df['label'] = label
                df['subject_id'] = f.stem.split('_')[1] if '_' in f.stem else f.stem
                all_dfs.append(df)
            except Exception as e:
                print(f"Error loading {f}: {e}")
    if not all_dfs:
        print("No MobiAct files loaded. Check data/raw/MobiAct/ directory structure.")
        return pd.DataFrame()
    return pd.concat(all_dfs, ignore_index=True)


if MOBIACT_DIR.exists():
    df_mobiact = load_mobiact_dataset(MOBIACT_DIR)
    print(f"MobiAct loaded: {len(df_mobiact):,} rows")
    print(df_mobiact['label'].value_counts())
else:
    print("MobiAct not available — using synthetic demo data")
    # Synthetic placeholder for development
    np.random.seed(42)
    n = 5000
    df_mobiact = pd.DataFrame({
        'acc_x': np.random.randn(n) * 0.3,
        'acc_y': np.random.randn(n) * 0.3 + 1.0,  # gravity component
        'acc_z': np.random.randn(n) * 0.3,
        'gyro_x': np.random.randn(n) * 0.05,
        'gyro_y': np.random.randn(n) * 0.05,
        'gyro_z': np.random.randn(n) * 0.05,
        'label': np.random.choice(['WAL', 'STD', 'FOL', 'BSC'], n, p=[0.4, 0.4, 0.1, 0.1]),
        'subject_id': np.random.randint(1, 10, n)
    })
    print(df_mobiact['label'].value_counts())

## 3. Class Imbalance Analysis

In [ ]:
label_counts = df_mobiact['label'].value_counts()
is_fall = label_counts.index.isin(FALL_LABELS_MOBIACT)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full label distribution
colors = ['#e74c3c' if l in FALL_LABELS_MOBIACT else '#3498db' for l in label_counts.index]
label_counts.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('MobiAct: Sample distribution by class')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Samples')
axes[0].tick_params(axis='x', rotation=45)

# Fall vs ADL aggregate
fall_total = label_counts[label_counts.index.isin(FALL_LABELS_MOBIACT)].sum()
adl_total = label_counts[~label_counts.index.isin(FALL_LABELS_MOBIACT)].sum()
axes[1].bar(['ADL', 'Fall'], [adl_total, fall_total], color=['#3498db', '#e74c3c'])
axes[1].set_title(f'Imbalance ratio: {adl_total/fall_total:.1f}:1 (ADL:Fall)')
axes[1].set_ylabel('Total samples')
for i, v in enumerate([adl_total, fall_total]):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/eda_falls_imbalance.png', bbox_inches='tight')
plt.show()
print(f"\nImbalance ratio ADL:Fall = {adl_total/fall_total:.1f}:1")
print("→ Justifies SMOTE + class_weight={0:1.0, 1:ratio}")

## 4. SVM Signal Visualization: Fall vs ADL

In [ ]:
def get_sensor_array(df_segment: pd.DataFrame) -> np.ndarray:
    """Extract [ax, ay, az, gx, gy, gz] array from a DataFrame segment."""
    accel_cols = [c for c in df_segment.columns if 'acc' in c.lower()][:3]
    gyro_cols = [c for c in df_segment.columns if 'gyro' in c.lower()][:3]
    return df_segment[accel_cols + gyro_cols].values.astype(np.float32)


fig, axes = plt.subplots(2, 2, figsize=(16, 10))

example_labels = {
    'FOL (Forward Fall)': 'FOL',
    'BSC (Backward Fall)': 'BSC',
    'WAL (Walking — ADL)': 'WAL',
    'STD (Standing — ADL)': 'STD',
}

for ax, (title, label) in zip(axes.flat, example_labels.items()):
    subset = df_mobiact[df_mobiact['label'] == label]
    if len(subset) == 0:
        ax.set_title(f'{title} — no data')
        continue
    # Take first subject's recording
    first_subj = subset['subject_id'].iloc[0]
    seg = subset[subset['subject_id'] == first_subj].head(300)
    sensor = get_sensor_array(seg)
    if sensor.shape[1] < 3:
        ax.set_title(f'{title} — insufficient columns')
        continue
    svm = compute_svm(sensor[:, :3])
    t = np.arange(len(svm)) / 50.0  # assume 50 Hz
    ax.plot(t, svm, label='SVM', linewidth=1.5, color='#2c3e50')
    ax.axhline(y=3.0, color='#e74c3c', linestyle='--', linewidth=1, label='3g threshold')
    ax.set_title(title)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('SVM (g)')
    ax.legend(fontsize=8)
    if label in FALL_LABELS_MOBIACT:
        peaks = detect_impact_peaks(svm, threshold_g=3.0)
        if len(peaks) > 0:
            ax.scatter(t[peaks], svm[peaks], color='#e74c3c', zorder=5, s=60, label='Impact peak')
            ax.legend(fontsize=8)

plt.suptitle('SVM signals: Falls vs ADL', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/eda_falls_svm_comparison.png', bbox_inches='tight')
plt.show()

## 5. Empirical SVM Threshold Analysis

In [ ]:
# Compute peak SVM per recording for falls and ADLs
peak_svms = {'Fall': [], 'ADL': []}

for (label, subj), group in df_mobiact.groupby(['label', 'subject_id']):
    sensor = get_sensor_array(group)
    if sensor.shape[1] < 3:
        continue
    svm = compute_svm(sensor[:, :3])
    peak = svm.max()
    category = 'Fall' if label in FALL_LABELS_MOBIACT else 'ADL'
    peak_svms[category].append(peak)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(peak_svms['ADL'], bins=50, alpha=0.6, color='#3498db', label=f'ADL (n={len(peak_svms["ADL"])})')
ax.hist(peak_svms['Fall'], bins=50, alpha=0.7, color='#e74c3c', label=f'Fall (n={len(peak_svms["Fall"])})')
ax.axvline(x=3.0, color='black', linestyle='--', label='3g threshold (proposed)')
ax.set_xlabel('Peak SVM (g)')
ax.set_ylabel('Count')
ax.set_title('Peak SVM distribution: Falls vs ADL')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/eda_falls_peak_svm_hist.png', bbox_inches='tight')
plt.show()

if peak_svms['Fall']:
    print(f"Fall peak SVM — median: {np.median(peak_svms['Fall']):.2f}g, 5th pct: {np.percentile(peak_svms['Fall'], 5):.2f}g")
if peak_svms['ADL']:
    print(f"ADL peak SVM  — median: {np.median(peak_svms['ADL']):.2f}g, 95th pct: {np.percentile(peak_svms['ADL'], 95):.2f}g")

## 6. Temporal Pattern: Impact → Post-fall Immobility

In [ ]:
# Show the 3-phase pattern of a fall: pre-fall activity → impact → post-fall immobility
fall_examples = df_mobiact[df_mobiact['label'].isin(FALL_LABELS_MOBIACT)]

if len(fall_examples) > 0:
    fig, axes = plt.subplots(len(FALL_LABELS_MOBIACT), 1, figsize=(14, 16))
    
    for ax, fall_type in zip(axes, FALL_LABELS_MOBIACT):
        subset = df_mobiact[df_mobiact['label'] == fall_type]
        if len(subset) == 0:
            ax.set_title(f'{fall_type} — no data')
            continue
        first_subj = subset['subject_id'].iloc[0]
        seg = subset[subset['subject_id'] == first_subj]
        sensor = get_sensor_array(seg)
        if sensor.shape[1] < 3:
            continue
        svm = compute_svm(sensor[:, :3])
        peaks = detect_impact_peaks(svm, threshold_g=2.5)
        t = np.arange(len(svm)) / 50.0
        
        ax.plot(t, svm, color='#2c3e50', linewidth=1)
        ax.axhline(y=3.0, color='#e74c3c', linestyle='--', alpha=0.7, label='3g threshold')
        if len(peaks) > 0:
            impact_idx = peaks[0]
            ax.axvspan(max(0, t[impact_idx]-0.5), min(t[-1], t[impact_idx]+0.5),
                       alpha=0.2, color='red', label='Impact zone')
            # Post-fall: 30 s after impact — show immobility
            post_start = impact_idx
            post_end = min(len(svm), impact_idx + 30*50)
            ax.axvspan(t[post_start], t[post_end-1], alpha=0.1, color='orange', label='30s post-fall check')
        
        ax.set_title(f'Fall type: {fall_type}')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('SVM (g)')
        ax.legend(fontsize=8, loc='upper right')
    
    plt.suptitle('Fall temporal pattern: impact → immobility', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/processed/eda_falls_temporal_pattern.png', bbox_inches='tight')
    plt.show()

## 7. SisFall — Elderly Subjects (60–75 años)

SisFall: 38 subjects (15 elderly 60–75, 23 young adults), 200 Hz → resample to 50 Hz.
Sensor placement: waist. Format: sensor1 (accel), sensor2 (accel), sensor3 (accel+gyro).

In [ ]:
SISFALL_DIR = DATA_RAW / 'SisFall'
SISFALL_HZ = 200


def load_sisfall_file(filepath: Path) -> np.ndarray:
    """Load SisFall .txt file. Format: 5 columns per sensor block.
    Uses sensor D1 (first accelerometer) + D3 (accelerometer + gyroscope).
    Returns (N, 6) array at 200 Hz.
    """
    try:
        data = np.loadtxt(filepath, delimiter=',', dtype=np.float32)
    except Exception:
        return None
    if data.ndim == 1:
        return None
    # SisFall columns depend on version — try to grab 6 columns
    if data.shape[1] >= 6:
        return data[:, :6]
    return None


if SISFALL_DIR.exists():
    sisfall_files = list(SISFALL_DIR.rglob('*.txt')) + list(SISFALL_DIR.rglob('*.csv'))
    print(f"SisFall: {len(sisfall_files)} files found")
    
    # Sample one file
    if sisfall_files:
        sample = load_sisfall_file(sisfall_files[0])
        if sample is not None:
            sample_50 = resample(sample, orig_hz=SISFALL_HZ, target_hz=50)
            svm = compute_svm(sample_50[:, :3])
            t = np.arange(len(svm)) / 50.0
            
            plt.figure(figsize=(12, 4))
            plt.plot(t, svm)
            plt.axhline(y=3.0, color='r', linestyle='--', label='3g threshold')
            plt.title(f'SisFall sample: {sisfall_files[0].name} (resampled to 50 Hz)')
            plt.xlabel('Time (s)')
            plt.ylabel('SVM (g)')
            plt.legend()
            plt.tight_layout()
            plt.show()
else:
    print("SisFall not available — download to data/raw/SisFall/")

## 8. Summary: Key EDA Findings

In [ ]:
print("=" * 60)
print("EDA Summary — Fall Detection Datasets")
print("=" * 60)

if peak_svms['Fall'] and peak_svms['ADL']:
    fall_p5 = np.percentile(peak_svms['Fall'], 5)
    adl_p95 = np.percentile(peak_svms['ADL'], 95)
    print(f"\n1. SVM Threshold Analysis:")
    print(f"   Fall 5th pct: {fall_p5:.2f}g")
    print(f"   ADL 95th pct: {adl_p95:.2f}g")
    optimal_threshold = (fall_p5 + adl_p95) / 2
    print(f"   Proposed threshold: {optimal_threshold:.2f}g (midpoint)")
    print(f"   Using 3.0g as conservative default.")

if peak_svms['Fall'] and peak_svms['ADL']:
    ratio = len(peak_svms['ADL']) / max(1, len(peak_svms['Fall']))
    print(f"\n2. Class Imbalance:")
    print(f"   ADL:Fall ratio = {ratio:.1f}:1")
    print(f"   → class_weight = {{0: 1.0, 1: {ratio:.1f}}} for model.fit()")
    print(f"   → SMOTE to oversample fall class before training")

print(f"\n3. Cascade design decision:")
print(f"   Stage 1: max(SVM_window) > 3g → trigger CNN")
print(f"   Stage 2: CNN sigmoid > threshold → fall confirmed")
print(f"   Stage 3: 30s immobility post-impact → escalate alert")

print(f"\n4. SisFall key fact:")
print(f"   15 elderly subjects (60-75 years) — unique for this segment")
print(f"   Resample from 200 Hz to 50 Hz for unified pipeline")
print(f"   Expect lower SVM peaks in elderly (slower falls)")